# Peut-on reconnaître un manchot Adélie à partir de sa fiche de terrain ?

Une équipe de la station Palmer reçoit une fiche de l’île Dream. Le bec mesure 39,8 mm de long et 18,4 mm de haut, la nageoire mesure 190 mm et la fiche indique une femelle. La balance était en panne. **L’équipe doit-elle classer ce manchot comme Adélie ?**

Tu disposes de fiches d’archives dont l’espèce a été identifiée. **Entraîne une régression logistique** pour distinguer les Adélie des autres manchots. À la fin, donne à l’équipe ta prédiction pour cette fiche.

## 1. Que sait-on au moment de la prédiction ?

Charge les archives avec la cellule ci-dessous. Repère la colonne qui contient la réponse connue. Relève les espèces présentes et le nombre de fiches de chaque espèce. Compte les valeurs manquantes par colonne.

Compare les ordres de grandeur de `bill_depth_mm` (hauteur du bec) et de `body_mass_g` (masse). Que signifierait un écart de 10 dans chacune de ces colonnes ?

In [2]:
# Exploimport pandas as pd
import pandas as pd
from IPython.display import display

manchots = pd.read_csv("penguins.csv")
display(manchots.head())
display(manchots["species"].value_counts().rename("Nombre de fiches"))
display(manchots.isna().sum().rename("Valeurs manquantes"))
display(manchots[["bill_depth_mm", "body_mass_g"]].describe().loc[["min", "mean", "max"]])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


species
Adelie       152
Gentoo       124
Chinstrap     68
Name: Nombre de fiches, dtype: int64

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
Name: Valeurs manquantes, dtype: int64

,bill_depth_mm,body_mass_g
min,13.10000,2700.000000
mean,17.15117,4201.754386
max,21.50000,6300.000000


## 2. Comment vérifier que le modèle reconnaît de nouveaux manchots ?

Construis la cible `y` avec la valeur 1 pour `Adelie` et 0 pour les deux autres espèces. Pour `X`, garde les quatre mesures physiques, l’île et le sexe. Explique ce qui se passerait si tu laissais la colonne `species` dans les entrées.

Affiche l’effectif et la proportion d’Adélie dans chaque lot. Garde le test pour le bilan final.

In [ ]:
# Prépare X et y, puis les lots d’apprentissage, de validation et de test.

# Importe la fonction qui répartit les données en plusieurs lots.
from sklearn.model_selection import train_test_split

# Liste les colonnes de mesures utilisées pour faire la prédiction.
numeriques = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Liste les colonnes qui contiennent des catégories.
categories = ["island", "sex"]

# Crée X, le tableau des données d’entrée, en copiant les six colonnes choisies.
X = manchots[numeriques + categories].copy()

# Convertit les colonnes catégorielles au type object pour le prétraitement.
X[categories] = X[categories].astype(object)

# Crée la cible y : 1 pour un manchot Adélie, 0 pour les autres espèces.
y = manchots["species"].eq("Adelie").astype(int)

# Sépare le jeu de test (environ 20 %) du reste des données (environ 80 %).
X_reste, X_test, y_reste, y_test = train_test_split(
    X, y,                  # Découpe les mesures et leurs étiquettes ensemble.
    test_size=0.20,         # Réserve 20 % du total pour le test.
    stratify=y,            # Garde une proportion similaire d’Adélie dans les deux lots.
    random_state=42,       # Permet de reproduire le même tirage.
)

# Découpe les données restantes en deux lots : entraînement et validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_reste, y_reste,      # Utilise uniquement les données restantes après le premier découpage.
    test_size=0.25,         # Prend un quart du reste pour la validation, soit environ 20 % du total.
    stratify=y_reste,      # Garde une proportion similaire d’Adélie dans les deux nouveaux lots.
    random_state=42,       # Permet de reproduire ce deuxième tirage.
)

# Associe le nom de chaque lot à ses étiquettes (0 ou 1).
lots = {"Apprentissage": y_train, "Validation": y_val, "Test": y_test}

# Parcourt les trois lots pour vérifier leur taille et leur composition.
for nom, cible in lots.items():
    # len compte toutes les fiches, sum compte les 1 et mean donne la proportion de 1.
    print(f"{nom} : {len(cible)} fiches, {cible.sum()} Adélie ({cible.mean():.2%})")

Apprentissage : 206 fiches, 91 Adélie (44.17%)
Validation : 69 fiches, 31 Adélie (44.93%)
Test : 69 fiches, 30 Adélie (43.48%)


## 3. Que représente une masse standardisée ?

Sur la seule colonne `body_mass_g` du lot d’apprentissage, ajuste un `SimpleImputer` qui remplace les masses manquantes par la médiane. Sur les masses ainsi complétées, ajuste un `StandardScaler`. Relève la médiane mémorisée, la moyenne et l’écart-type utilisés par le scaler.

Choisis la première fiche de validation dont la masse est renseignée. Calcule sa masse standardisée à la main avec les paramètres appris, puis vérifie ton résultat avec `transform`. Que signifierait une valeur standardisée de −1 pour un manchot ?

In [7]:
# Affiche la masse en grammes, ton calcul et le résultat de transform.
# Importe les outils pour compléter les valeurs manquantes et standardiser les masses.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Sélectionne la masse dans le lot d’apprentissage. Les doubles crochets gardent un tableau à une colonne.
masses_train = X_train[["body_mass_g"]]

# Crée un outil qui remplace les masses manquantes par la médiane.
imputer = SimpleImputer(strategy="median")
# Calcule la médiane sur les masses d’apprentissage, puis l’utilise pour compléter les masses manquantes.
masses_train_completees = imputer.fit_transform(masses_train)
# Récupère la médiane mémorisée pour l’unique colonne.
mediane = imputer.statistics_[0]

# Crée un outil qui centre les masses autour de zéro et les exprime en écarts-types.
scaler = StandardScaler()
# Calcule la moyenne et l’écart-type à partir des masses d’apprentissage complétées.
scaler.fit(masses_train_completees)
# Récupère la moyenne mémorisée par le scaler.
moyenne = scaler.mean_[0]
# Récupère l’écart-type utilisé pour diviser les écarts à la moyenne.
ecart_type = scaler.scale_[0]

# Affiche les trois paramètres appris, en grammes.
print(f"Médiane : {mediane:.2f} g")
print(f"Moyenne : {moyenne:.2f} g")
print(f"Écart-type : {ecart_type:.2f} g")

# Garde les fiches de validation dont la masse est renseignée, puis prend la première.
fiche_val = X_val[["body_mass_g"]].dropna().iloc[[0]]
# Extrait la valeur de masse du tableau à une ligne et une colonne.
masse = fiche_val.iloc[0, 0]

# Applique directement la formule : (masse - moyenne) / écart-type.
masse_standardisee = (masse - moyenne) / ecart_type
# Applique l’imputation déjà apprise. La masse reste identique puisqu’elle est renseignée.
masse_val_completee = imputer.transform(fiche_val)
# Vérifie le calcul avec les paramètres déjà appris par le scaler.
masse_transformee = scaler.transform(masse_val_completee)[0, 0]

# Affiche la masse brute et les deux résultats à comparer.
print(f"Masse de la fiche : {masse:.2f} g")
print(f"Calcul direct : {masse_standardisee:.4f}")
print(f"Résultat de transform : {masse_transformee:.4f}")



Médiane : 4050.00 g
Moyenne : 4208.86 g
Écart-type : 791.78 g
Masse de la fiche : 5950.00 g
Calcul direct : 2.1990
Résultat de transform : 2.1990


## 4. Comment préparer une fiche entière ?

Construis un `ColumnTransformer` qui applique le traitement adapté à chaque colonne.

| Colonnes | Opérations à enchaîner |
|---|---|
| Longueur et hauteur du bec, longueur de la nageoire, masse | Compléter chaque mesure manquante avec sa médiane d’apprentissage, puis appliquer `StandardScaler`. |
| Île et sexe | Remplacer les valeurs manquantes par `Non renseigné`, puis utiliser `OneHotEncoder(handle_unknown="ignore")`. |

Ajuste cette préparation sur les entrées d’apprentissage. Transforme une fiche de validation, puis affiche les noms et les valeurs des colonnes obtenues. Retrouve la masse standardisée et la colonne qui indique l’île de cette fiche.

In [11]:
# Assemble les deux traitements, puis inspecte une fiche transformée.
# Importe les outils pour appliquer un traitement différent à chaque groupe de colonnes.
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Enchaîne les deux opérations pour les quatre colonnes numériques.
traitement_numerique = make_pipeline(
    SimpleImputer(strategy="median"),  # Remplace les valeurs manquantes par la médiane de chaque colonne.
    StandardScaler(),                 # Centre et standardise chaque colonne séparément.
)

# Enchaîne les deux opérations pour l’île et le sexe.
traitement_categoriel = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="Non renseigné"),  # Complète les valeurs manquantes.
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),      # Crée des colonnes de 0 et de 1.
)

# Associe chaque traitement aux listes de colonnes définies à la question 2.
preparation = ColumnTransformer([
    ("num", traitement_numerique, numeriques),     # Traite les quatre mesures.
    ("cat", traitement_categoriel, categories),   # Traite l’île et le sexe.
])

# Apprend les médianes, les paramètres de standardisation et les catégories sur X_train.
preparation.fit(X_train)

# Prend la première fiche de validation avec une masse connue et garde toutes ses colonnes.
fiche_val = X_val.loc[X_val["body_mass_g"].notna()].iloc[[0]]

# Prépare cette fiche avec les paramètres appris sur le lot d’apprentissage.
valeurs = preparation.transform(fiche_val)
# Récupère les noms des colonnes produites, avec les préfixes num__ et cat__.
noms = preparation.get_feature_names_out()
# Affiche une ligne par colonne transformée pour lire facilement son nom et sa valeur.
fiche_preparee = pd.DataFrame(valeurs, columns=noms, index=fiche_val.index)
display(fiche_preparee.T)

# Retrouve la masse standardisée dans la colonne numérique correspondante.
print(f"Masse standardisée : {fiche_preparee['num__body_mass_g'].iloc[0]:.4f}")

# Récupère uniquement les colonnes créées pour l’île.
colonnes_ile = fiche_preparee.filter(like="cat__island__").iloc[0] if False else fiche_preparee.filter(like="cat__island_").iloc[0]
# Affiche la colonne qui vaut 1 et indique donc l’île de la fiche.
print("Colonne de l’île :", ", ".join(colonnes_ile[colonnes_ile.eq(1)].index))

,263
num__bill_length_mm,1.037764
num__bill_depth_mm,-0.691442
num__flipper_length_mm,2.017990
num__body_mass_g,2.199012
cat__island_Biscoe,1.000000
cat__island_Dream,0.000000
cat__island_Torgersen,0.000000
cat__sex_Non renseigné,0.000000
cat__sex_female,0.000000
cat__sex_male,1.000000


Masse standardisée : 2.1990
Colonne de l’île : cat__island_Biscoe


## 5. La standardisation change-t-elle les prédictions ?

Construis deux pipelines qui acceptent les fiches brutes et se terminent par `LogisticRegression(max_iter=5000)`. Le premier utilise la préparation de l’étape 4. Pour le second, retire uniquement le scaler de la branche numérique. Garde l’imputation, l’encodage, le modèle et les lots identiques.

Entraîne les deux pipelines sur l’apprentissage. Pour chacun, compte les fiches de validation mal classées et calcule la proportion de prédictions correctes. Retiens celui qui fait le moins d’erreurs. En cas d’égalité, garde la version standardisée. Explique ton choix avec les résultats observés.

In [12]:
# Entraîne les deux pipelines et compare leurs prédictions sur la validation.
# Importe les outils pour copier la préparation, construire les pipelines et mesurer leur performance.
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

# Copie la configuration de la question 4 en conservant la standardisation.
preparation_avec_scaler = clone(preparation)

# Crée une deuxième copie indépendante de la préparation.
preparation_sans_scaler = clone(preparation)
# Dans la branche « num », passe directement de l’imputation au modèle.
preparation_sans_scaler.set_params(num__standardscaler="passthrough")

# Construit le pipeline qui standardise les mesures avant la régression logistique.
pipeline_avec_scaler = make_pipeline(
    preparation_avec_scaler,
    LogisticRegression(max_iter=5000),
)

# Construit le pipeline qui utilise les mesures complétées dans leurs unités d’origine.
pipeline_sans_scaler = make_pipeline(
    preparation_sans_scaler,
    LogisticRegression(max_iter=5000),
)

# Entraîne les deux pipelines avec les mêmes fiches et les mêmes réponses.
pipeline_avec_scaler.fit(X_train, y_train)
pipeline_sans_scaler.fit(X_train, y_train)

# Prédit la classe de chaque fiche brute du lot de validation.
pred_avec = pipeline_avec_scaler.predict(X_val)
pred_sans = pipeline_sans_scaler.predict(X_val)

# Compare chaque prédiction à la vraie classe et compte les erreurs.
erreurs_avec = (pred_avec != y_val).sum()
erreurs_sans = (pred_sans != y_val).sum()

# Calcule la proportion de fiches correctement classées dans chaque version.
score_avec = accuracy_score(y_val, pred_avec)
score_sans = accuracy_score(y_val, pred_sans)

# Compte les fiches qui reçoivent une classe différente selon la version.
predictions_differentes = (pred_avec != pred_sans).sum()

# Affiche les résultats de validation.
print(f"Avec standardisation   : {erreurs_avec} erreurs, {score_avec:.2%} de bonnes prédictions")
print(f"Sans standardisation   : {erreurs_sans} erreurs, {score_sans:.2%} de bonnes prédictions")
print(f"Fiches dont la prédiction change : {predictions_differentes}")

# Garde le meilleur pipeline. Le symbole <= donne la priorité à la version standardisée en cas d’égalité.
if erreurs_avec <= erreurs_sans:
    pipeline_retenu = pipeline_avec_scaler
    if erreurs_avec == erreurs_sans:
        print(f"Je retiens la version standardisée : les deux versions font {erreurs_avec} erreurs.")
    else:
        print(f"Je retiens la version standardisée : {erreurs_avec} erreurs contre {erreurs_sans}.")
else:
    pipeline_retenu = pipeline_sans_scaler
    print(f"Je retiens la version sans standardisation : {erreurs_sans} erreurs contre {erreurs_avec}.")

Avec standardisation   : 0 erreurs, 100.00% de bonnes prédictions
Sans standardisation   : 0 erreurs, 100.00% de bonnes prédictions
Fiches dont la prédiction change : 0
Je retiens la version standardisée : les deux versions font 0 erreurs.


## 6. Quelle réponse donner à l’équipe de terrain ?

Évalue une seule fois le pipeline retenu sur le test. Compte les Adélie manqués et les autres manchots classés à tort comme Adélie. Vérifie que la somme donne ton nombre total d’erreurs.

La fiche du début est reproduite ci-dessous. Transmets-la directement au pipeline retenu avec `predict`. Termine par deux phrases pour l’équipe. Donne la classe prédite pour ce manchot, puis situe la fiabilité de la décision à partir des erreurs observées sur le test.

In [19]:
# Évalue le pipeline retenu sur le lot de test.
# Prédit une seule fois les classes des manchots du lot de test.
pred_test = pipeline_retenu.predict(X_test)

# Compte les vrais Adélie (1) que le modèle a classés comme autres (0).
adelie_manques = ((y_test == 1) & (pred_test == 0)).sum()

# Compte les autres manchots (0) que le modèle a classés comme Adélie (1).
autres_classes_adelie = ((y_test == 0) & (pred_test == 1)).sum()

# Compte toutes les prédictions différentes de la vraie classe.
erreurs_test = (pred_test != y_test).sum()

# Vérifie que les deux types d’erreurs couvrent toutes les erreurs du test.
assert adelie_manques + autres_classes_adelie == erreurs_test

# Affiche le détail des erreurs et leur total.
print(f"Adélie manqués : {adelie_manques}")
print(f"Autres manchots classés à tort comme Adélie : {autres_classes_adelie}")
print(f"Total : {erreurs_test} erreurs sur {len(y_test)} fiches")


Adélie manqués : 0
Autres manchots classés à tort comme Adélie : 0
Total : 0 erreurs sur 69 fiches


In [20]:
fiche_dream = pd.DataFrame([{
    "bill_length_mm": 39.8,
    "bill_depth_mm": 18.4,
    "flipper_length_mm": 190,
    "body_mass_g": float("nan"),
    "island": "Dream",
    "sex": "female",
}])

# Fais prédire la classe de fiche_dream par le pipeline retenu.
# Transmet la fiche brute au pipeline, qui prend en charge la masse manquante.
prediction_dream = pipeline_retenu.predict(fiche_dream)[0]

# Convertit la prédiction numérique en une classe lisible.
classe_predite = "Adélie" if prediction_dream == 1 else "autre espèce (non-Adélie)"

# Affiche les deux phrases à transmettre à l’équipe.
print(f"Pour le manchot observé sur l’île Dream, le modèle prédit la classe « {classe_predite} ».")
print(
    f"Sur les {len(y_test)} fiches du test, le modèle a fait {erreurs_test} erreurs "
    f"({erreurs_test / len(y_test):.1%}), dont {adelie_manques} Adélie manqués "
    f"et {autres_classes_adelie} autres manchots classés à tort comme Adélie."
)





Pour le manchot observé sur l’île Dream, le modèle prédit la classe « Adélie ».
Sur les 69 fiches du test, le modèle a fait 0 erreurs (0.0%), dont 0 Adélie manqués et 0 autres manchots classés à tort comme Adélie.


**Ta réponse à l’équipe**  
…